In [1]:
%pip install pandas numpy matplotlib seaborn openpyxl


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:

# ============================================================
# ZEROQ AI - NOTEBOOK 01
# DATA LOADING AND INITIAL DATA QUALITY AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")


# ------------------------------------------------------------
# 2. DEFINE PROJECT PATHS
# ------------------------------------------------------------

PROJECT_PATH = Path(
    r"D:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\model_datasets"
)

BASE_PATH = PROJECT_PATH / "raw_data"

PREPROCESSING_PATH = PROJECT_PATH / "preprocessing"
PROCESSED_DATA_PATH = PROJECT_PATH / "processed_data"
VISUALIZATION_PATH = PROJECT_PATH / "visualizations"

REPORT_PATH = PREPROCESSING_PATH / "reports"

# Create required directories
for folder in [
    PREPROCESSING_PATH,
    PROCESSED_DATA_PATH,
    VISUALIZATION_PATH,
    REPORT_PATH
]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nPROJECT CONFIGURATION")
print("-" * 60)
print("Project path:", PROJECT_PATH)
print("Raw data path:", BASE_PATH)
print("Raw data folder exists:", BASE_PATH.exists())


# ------------------------------------------------------------
# 3. DEFINE DATASET FILENAMES
# ------------------------------------------------------------

DATASET_FILES = {
    "products": "products.csv",
    "customers": "customers.csv",
    "facilities": "facilities.csv",
    "production_lines": "production_lines.csv",
    "materials": "materials.csv",
    "production_batches": "production_batches.csv",
    "complaints": "complaints.csv",
    "quality_inspections": "quality_inspections.csv",
    "investigations": "investigations.csv",
    "corrective_actions": "corrective_actions.csv",
    "data_dictionary": "data_dictionary.csv"
}

print("\nTotal expected datasets:", len(DATASET_FILES))

from pathlib import Path

# ------------------------------------------------
# AUTOMATIC PROJECT AND RAW DATA FOLDER DETECTION
# ------------------------------------------------

print("Current working directory:", Path.cwd())

# Expected CSV filenames
required_files = {
    "products.csv",
    "customers.csv",
    "facilities.csv",
    "production_lines.csv",
    "materials.csv",
    "production_batches.csv",
    "complaints.csv",
    "quality_inspections.csv",
    "investigations.csv",
    "corrective_actions.csv",
    "data_dictionary.csv"
}

# Search current directory and its parent directories
search_roots = [Path.cwd(), *Path.cwd().parents]

matching_folders = []

for root in search_roots:

    for folder in [root] + [
        p for p in root.rglob("*") if p.is_dir()
    ]:

        available_files = {
            file.name for file in folder.glob("*.csv")
        }

        if required_files.issubset(available_files):
            matching_folders.append(folder)

    if matching_folders:
        break


# Identify the dataset directory
if len(matching_folders) == 1:

    BASE_PATH = matching_folders[0]

    print("\nCSV folder identified successfully:")
    print(BASE_PATH)

elif len(matching_folders) > 1:

    print("\nMultiple folders contain all 11 CSV files:")
    for folder in matching_folders:
        print(folder)

    raise RuntimeError(
        "Multiple matching folders found. Select the correct one."
    )

else:

    raise FileNotFoundError(
        "Could not locate a folder containing all 11 CSV files. "
        "Check the VS Code workspace location."
    )


# Identify project directory
PROJECT_PATH = BASE_PATH.parent

# Define remaining directories
PREPROCESSING_PATH = PROJECT_PATH / "preprocessing"
PROCESSED_DATA_PATH = PROJECT_PATH / "processed_data"
VISUALIZATION_PATH = PROJECT_PATH / "visualizations"
REPORT_PATH = PREPROCESSING_PATH / "reports"

# Create required directories
for folder in [
    PREPROCESSING_PATH,
    PROCESSED_DATA_PATH,
    VISUALIZATION_PATH,
    REPORT_PATH
]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nProject directory:", PROJECT_PATH)
print("Raw data directory:", BASE_PATH)
print("Raw data folder exists:", BASE_PATH.exists())



# ------------------------------------------------------------
# 4. LOAD ALL CSV DATASETS
# ------------------------------------------------------------

datasets = {}
missing_files = []

if not BASE_PATH.exists():
    raise FileNotFoundError(
        f"Raw data directory does not exist: {BASE_PATH}"
    )

for dataset_name, filename in DATASET_FILES.items():

    file_path = BASE_PATH / filename

    if file_path.exists():

        try:
            df = pd.read_csv(file_path)

        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding="latin1")

        datasets[dataset_name] = df

        print(
            f"{dataset_name:25} | "
            f"Loaded successfully | "
            f"Shape: {df.shape}"
        )

    else:
        missing_files.append(filename)
        print(f"{filename} NOT FOUND")

print("\nDATASET LOADING SUMMARY")
print("-" * 60)
print("Expected datasets:", len(DATASET_FILES))
print("Successfully loaded:", len(datasets))
print("Missing files:", len(missing_files))

if missing_files:
    print("\nMissing files:")
    for filename in missing_files:
        print("-", filename)

if not datasets:
    raise ValueError("No datasets were loaded. Check the raw_data folder.")


# ------------------------------------------------------------
# 5. DATASET DIMENSION ANALYSIS
# ------------------------------------------------------------

dataset_summary = []

for name, df in datasets.items():

    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 3
        )
    })

dataset_summary_df = pd.DataFrame(dataset_summary)

print("\nDATASET DIMENSION SUMMARY")
display(dataset_summary_df)


# ------------------------------------------------------------
# 6. COLUMN NAMES AND DATA TYPE ANALYSIS
# ------------------------------------------------------------

schema_reports = []

for name, df in datasets.items():

    for column in df.columns:

        schema_reports.append({
            "Dataset": name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Non_Null_Count": int(df[column].notna().sum()),
            "Null_Count": int(df[column].isna().sum()),
            "Unique_Values": int(df[column].nunique(dropna=True))
        })

schema_report_df = pd.DataFrame(schema_reports)

print("\nDATASET SCHEMA ANALYSIS")
display(schema_report_df)


# ------------------------------------------------------------
# 7. MISSING VALUE ANALYSIS
# ------------------------------------------------------------

missing_value_reports = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        missing_percentage = (
            (missing_count / len(df)) * 100
            if len(df) > 0 else 0
        )

        missing_value_reports.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": int(missing_count),
            "Missing_Percentage": round(missing_percentage, 2)
        })

missing_report_df = pd.DataFrame(
    missing_value_reports,
    columns=[
        "Dataset",
        "Column",
        "Missing_Count",
        "Missing_Percentage"
    ]
)

if not missing_report_df.empty:

    missing_report_df = missing_report_df.sort_values(
        by="Missing_Percentage",
        ascending=False
    ).reset_index(drop=True)

print("\nMISSING VALUE ANALYSIS")

display(
    missing_report_df[
        missing_report_df["Missing_Count"] > 0
    ]
)


# ------------------------------------------------------------
# 8. DUPLICATE RECORD ANALYSIS
# ------------------------------------------------------------

duplicate_reports = []

for name, df in datasets.items():

    duplicate_count = int(df.duplicated().sum())

    duplicate_percentage = (
        (duplicate_count / len(df)) * 100
        if len(df) > 0 else 0
    )

    duplicate_reports.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Duplicate_Rows": duplicate_count,
        "Duplicate_Percentage": round(duplicate_percentage, 2)
    })

duplicate_report_df = pd.DataFrame(
    duplicate_reports,
    columns=[
        "Dataset",
        "Total_Rows",
        "Duplicate_Rows",
        "Duplicate_Percentage"
    ]
)

print("\nDUPLICATE RECORD ANALYSIS")
display(duplicate_report_df)


# ------------------------------------------------------------
# 9. PRIMARY KEY UNIQUENESS ANALYSIS
# ------------------------------------------------------------

PRIMARY_KEYS = {
    "products": "product_id",
    "customers": "customer_id",
    "facilities": "facility_id",
    "production_lines": "production_line_id",
    "materials": "material_id",
    "production_batches": "batch_id",
    "complaints": "complaint_id",
    "quality_inspections": "inspection_id",
    "investigations": "investigation_id",
    "corrective_actions": "action_id"
}

primary_key_reports = []

for dataset_name, primary_key in PRIMARY_KEYS.items():

    if dataset_name not in datasets:
        continue

    df = datasets[dataset_name]

    if primary_key not in df.columns:

        primary_key_reports.append({
            "Dataset": dataset_name,
            "Primary_Key": primary_key,
            "Missing_Key_Column": True,
            "Null_Keys": None,
            "Duplicate_Keys": None,
            "Status": "COLUMN NOT FOUND"
        })

        continue

    null_count = int(df[primary_key].isna().sum())
    duplicate_count = int(df[primary_key].duplicated().sum())

    status = (
        "PASS"
        if null_count == 0 and duplicate_count == 0
        else "CHECK REQUIRED"
    )

    primary_key_reports.append({
        "Dataset": dataset_name,
        "Primary_Key": primary_key,
        "Missing_Key_Column": False,
        "Null_Keys": null_count,
        "Duplicate_Keys": duplicate_count,
        "Status": status
    })

primary_key_report_df = pd.DataFrame(
    primary_key_reports,
    columns=[
        "Dataset",
        "Primary_Key",
        "Missing_Key_Column",
        "Null_Keys",
        "Duplicate_Keys",
        "Status"
    ]
)

print("\nPRIMARY KEY VALIDATION")
display(primary_key_report_df)


# ------------------------------------------------------------
# 10. DATE RANGE ANALYSIS
# ------------------------------------------------------------

date_reports = []

for name, df in datasets.items():

    for column in df.columns:

        if (
            "date" in column.lower()
            or column.lower().endswith("_at")
        ):

            parsed_dates = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            valid_dates = parsed_dates.dropna()

            date_reports.append({
                "Dataset": name,
                "Date_Column": column,
                "Valid_Dates": len(valid_dates),
                "Invalid_or_Missing_Dates": int(
                    parsed_dates.isna().sum()
                ),
                "Minimum_Date": (
                    valid_dates.min()
                    if not valid_dates.empty else None
                ),
                "Maximum_Date": (
                    valid_dates.max()
                    if not valid_dates.empty else None
                )
            })

date_report_df = pd.DataFrame(
    date_reports,
    columns=[
        "Dataset",
        "Date_Column",
        "Valid_Dates",
        "Invalid_or_Missing_Dates",
        "Minimum_Date",
        "Maximum_Date"
    ]
)

print("\nDATE RANGE ANALYSIS")
display(date_report_df)


# ------------------------------------------------------------
# 11. NUMERICAL SUMMARY ANALYSIS
# ------------------------------------------------------------

numerical_reports = []

for name, df in datasets.items():

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for column in numeric_columns:

        numerical_reports.append({
            "Dataset": name,
            "Column": column,
            "Count": df[column].count(),
            "Mean": df[column].mean(),
            "Standard_Deviation": df[column].std(),
            "Minimum": df[column].min(),
            "25_Percentile": df[column].quantile(0.25),
            "Median": df[column].median(),
            "75_Percentile": df[column].quantile(0.75),
            "Maximum": df[column].max()
        })

numerical_report_df = pd.DataFrame(numerical_reports)

print("\nNUMERICAL SUMMARY")
display(numerical_report_df)


# ------------------------------------------------------------
# 12. CATEGORICAL VALUE ANALYSIS
# ------------------------------------------------------------

categorical_reports = []

for name, df in datasets.items():

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        mode_values = df[column].mode(dropna=True)

        most_common_value = (
            mode_values.iloc[0]
            if not mode_values.empty else None
        )

        categorical_reports.append({
            "Dataset": name,
            "Column": column,
            "Unique_Values": int(df[column].nunique(dropna=True)),
            "Most_Common_Value": most_common_value
        })

categorical_report_df = pd.DataFrame(
    categorical_reports,
    columns=[
        "Dataset",
        "Column",
        "Unique_Values",
        "Most_Common_Value"
    ]
)

print("\nCATEGORICAL VALUE ANALYSIS")
display(categorical_report_df)


# ------------------------------------------------------------
# 13. PREVIEW SAMPLE RECORDS
# ------------------------------------------------------------

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print("SAMPLE RECORDS:", name.upper())
    print("=" * 80)

    display(df.head(5))


# ------------------------------------------------------------
# 14. SAVE ALL AUDIT REPORTS TO EXCEL
# ------------------------------------------------------------

REPORT_FILE = REPORT_PATH / "initial_data_audit.xlsx"

with pd.ExcelWriter(
    REPORT_FILE,
    engine="openpyxl"
) as writer:

    dataset_summary_df.to_excel(
        writer,
        sheet_name="Dataset_Summary",
        index=False
    )

    schema_report_df.to_excel(
        writer,
        sheet_name="Schema_Analysis",
        index=False
    )

    missing_report_df.to_excel(
        writer,
        sheet_name="Missing_Values",
        index=False
    )

    duplicate_report_df.to_excel(
        writer,
        sheet_name="Duplicate_Rows",
        index=False
    )

    primary_key_report_df.to_excel(
        writer,
        sheet_name="Primary_Key_Audit",
        index=False
    )

    date_report_df.to_excel(
        writer,
        sheet_name="Date_Range_Audit",
        index=False
    )

    numerical_report_df.to_excel(
        writer,
        sheet_name="Numerical_Summary",
        index=False
    )

    categorical_report_df.to_excel(
        writer,
        sheet_name="Categorical_Audit",
        index=False
    )

print("\nAudit report saved successfully:")
print(REPORT_FILE)


# ------------------------------------------------------------
# 15. FINAL DATA AUDIT SUMMARY
# ------------------------------------------------------------

datasets_with_missing = (
    missing_report_df[
        missing_report_df["Missing_Count"] > 0
    ]["Dataset"].nunique()
    if not missing_report_df.empty else 0
)

datasets_with_duplicates = (
    duplicate_report_df[
        duplicate_report_df["Duplicate_Rows"] > 0
    ]["Dataset"].nunique()
    if not duplicate_report_df.empty else 0
)

primary_keys_requiring_attention = (
    len(
        primary_key_report_df[
            primary_key_report_df["Status"] != "PASS"
        ]
    )
    if not primary_key_report_df.empty else 0
)

print("\n")
print("=" * 65)
print("ZEROQ AI - INITIAL DATA AUDIT SUMMARY")
print("=" * 65)

print(f"Expected datasets                  : {len(DATASET_FILES)}")
print(f"Successfully loaded                : {len(datasets)}")
print(f"Missing files                      : {len(missing_files)}")
print(f"Datasets with missing values       : {datasets_with_missing}")
print(f"Datasets with duplicate rows       : {datasets_with_duplicates}")
print(f"Primary keys requiring attention   : {primary_keys_requiring_attention}")

print("-" * 65)

if len(datasets) == len(DATASET_FILES) and not missing_files:
    print("DATASET LOADING STATUS: SUCCESS")
else:
    print("DATASET LOADING STATUS: CHECK REQUIRED")

print("AUDIT REPORT:", REPORT_FILE)

print("=" * 65)
print("INITIAL DATA AUDIT COMPLETED")
print("=" * 65)


Libraries imported successfully.

PROJECT CONFIGURATION
------------------------------------------------------------
Project path: D:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\model_datasets
Raw data path: D:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\model_datasets\raw_data
Raw data folder exists: False

Total expected datasets: 11
Current working directory: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing

CSV folder identified successfully:
d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\raw_data

Project directory: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training
Raw data directory: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\raw_data
Raw data folder exists: True
products                  | Loaded successfully | Shape: (20, 6)
customers                 | Loaded successfully | Shape: (500, 5)
facilities                | Loaded successfully | Shape: (5, 4)
production_lines          | Loaded successfully | Shape: (15

,Dataset,Rows,Columns,Memory_MB
0,products,20,6,0.006
1,customers,500,5,0.127
2,facilities,5,4,0.001
3,production_lines,15,6,0.005
4,materials,40,6,0.013
5,production_batches,24000,20,17.746
6,complaints,6000,14,4.891
7,quality_inspections,37245,11,19.311
8,investigations,4000,10,2.640
9,corrective_actions,3500,13,3.009



DATASET SCHEMA ANALYSIS


,Dataset,Column,Data_Type,Non_Null_Count,Null_Count,Unique_Values
0,products,product_id,object,20,0,20
1,products,product_name,object,20,0,20
2,products,product_category,object,20,0,10
3,products,storage_condition,object,20,0,2
4,products,shelf_life_days,int64,20,0,6
5,products,standard_unit,object,20,0,1
6,customers,customer_id,object,500,0,500
7,customers,customer_type,object,500,0,4
8,customers,region,object,500,0,5
9,customers,customer_tenure_years,int64,500,0,25



MISSING VALUE ANALYSIS


,Dataset,Column,Missing_Count,Missing_Percentage


... (29 outputs left out)

,action_id,investigation_id,complaint_id,action_type,corrective_action,preventive_action,responsible_department,action_owner_id,action_implementation_date,effectiveness_review_date,effectiveness_status,recurrence_observed_90d,action_closure_status
0,ACT000001,INV000001,CMP000001,Preventive,Calibrate sealing temperature controller,Add hourly seal integrity verification,Production,EMP0119,2015-04-15,2015-06-26,Effective,0,Pending Verification
1,ACT000002,INV000003,CMP000003,Preventive,Recalibrate filling machine,Increase in-process weight sampling,Production,EMP0107,2009-10-22,2009-11-09,Effective,0,Closed
2,ACT000003,INV000004,CMP000004,Corrective and Preventive,Review and correct the identified process condition,Add a preventive verification to the relevant quality checklist,Production,EMP0058,2018-03-06,2018-04-12,Ineffective,1,Closed
3,ACT000004,INV000005,CMP000005,Corrective,Review and correct the identified process condition,Add a preventive verification to the relevant quality checklist,Procurement,EMP0034,2014-06-18,2014-08-05,Effective,0,Closed
4,ACT000005,INV000006,CMP000006,Preventive,Review and correct the identified process condition,Add a preventive verification to the relevant quality checklist,Procurement,EMP0005,2026-02-02,2026-04-12,Partially Effective,1,Pending Verification



SAMPLE RECORDS: DATA_DICTIONARY


,file_name,column_name,description,relationship_or_usage
0,products.csv,product_id,Unique product key,Primary key
1,customers.csv,customer_id,Unique customer key,Primary key
2,facilities.csv,facility_id,Unique manufacturing facility key,Primary key
3,production_lines.csv,production_line_id,Unique production line key,Primary key; facility_id references facilities
4,materials.csv,material_id,Unique raw or packaging material key,Primary key



Audit report saved successfully:
d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports\initial_data_audit.xlsx


ZEROQ AI - INITIAL DATA AUDIT SUMMARY
Expected datasets                  : 11
Successfully loaded                : 11
Missing files                      : 0
Datasets with missing values       : 0
Datasets with duplicate rows       : 0
Primary keys requiring attention   : 0
-----------------------------------------------------------------
DATASET LOADING STATUS: SUCCESS
AUDIT REPORT: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports\initial_data_audit.xlsx
INITIAL DATA AUDIT COMPLETED
